# 00 - Extraction des données

**But** : lire le gros fichier PGN de Lichess et en extraire les parties qu'on utilise dans un fichier Parquet,
rapide à relire dans les notebooks suivants.

- Source : `lichess_db_standard_rated_2026-09.pgn.zst` (28 Go compressé, toutes les parties classées de septembre 2026),
  téléchargé automatiquement depuis [database.lichess.org](https://database.lichess.org/) dans `Data/` s'il n'y est pas déjà.
- Sortie : `Data/blitz_180_raw.parquet`, 1 ligne par partie, utilisé par les notebooks 01 et 02.

**Choix** :
- les **2M premières parties** du fichier (≈ 24 h du 1er septembre) : largement assez de données pour apprendre ;
- seulement les cadences **180+0** et **180+2** (Blitz 3 min) : l'Elo dépend de la cadence et la pendule n'est
  comparable qu'à temps égal (cf. notebook 02).

Durée : ~4 min pour l'extraction (+ le téléchargement la première fois). À ne relancer que si on change ces choix.

In [1]:
import itertools
import time
import warnings
from pathlib import Path

import polars as pl

from utils.pgn import download_pgn, open_pgn, read_games

PGN_URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2026-09.pgn.zst"
PGN_PATH = Path("../Data/lichess_db_standard_rated_2026-09.pgn.zst")
RAW_PATH = Path("../Data/blitz_180_raw.parquet")

N_READ = 2_000_000
CADENCES = {"180+0", "180+2"}

## 0. Téléchargement

⚠️ **Le fichier fait ~28 Go.** Le téléchargement peut prendre plus d'une demi-heure selon la connexion, et il faut
28 Go libres sur le disque. Il n'est fait qu'une fois : si le fichier est déjà dans `Data/`, cette cellule ne fait rien.

In [2]:
if PGN_PATH.exists():
    print(f"{PGN_PATH} déjà présent ({PGN_PATH.stat().st_size / 1e9:.1f} Go) : pas de téléchargement.")
else:
    warnings.warn("Téléchargement de ~28 Go : long, et 28 Go de disque nécessaires.")
    download_pgn(PGN_URL, PGN_PATH)
    print(f"Terminé : {PGN_PATH}")

..\Data\lichess_db_standard_rated_2026-09.pgn.zst déjà présent (29.2 Go) : pas de téléchargement.


## 1. À quoi ressemble le fichier brut ?

Ce n'est pas un tableau mais du texte (format PGN). Chaque partie =
- des lignes d'en-têtes `[Clé "valeur"]` (joueurs, Elo, résultat, cadence…) ;
- une ligne vide ;
- **une seule ligne** avec tous les coups : `1. e4 { [%clk 0:03:00] } 1... c5 { [%clk 0:03:00] } ...`

`1.` = coup des Blancs, `1...` = coup des Noirs, `%clk` = temps restant à la pendule, `%eval` = évaluation de
l'ordinateur (seulement si quelqu'un a demandé l'analyse de la partie). Le fichier est lu **en streaming** :
on ne décompresse jamais les 28 Go.

In [3]:
with open_pgn(PGN_PATH) as f:
    for line in itertools.islice(f, 22):
        print(line[:300], end="")

[Event "Rated Bullet game"]
[Site "https://lichess.org/bcRJNu4b"]
[Date "2026.09.01"]
[Round "-"]
[White "Mycoba"]
[Black "Sentenced222"]
[Result "0-1"]
[UTCDate "2026.09.01"]
[UTCTime "00:00:20"]
[WhiteElo "1058"]
[BlackElo "1029"]
[WhiteRatingDiff "-7"]
[BlackRatingDiff "+11"]
[ECO "B21"]
[Opening "Sicilian Defense: Smith-Morra Gambit"]
[TimeControl "120+1"]
[Termination "Normal"]

1. e4 { [%clk 0:02:00] } 1... c5 { [%clk 0:02:00] } 2. d4 { [%clk 0:02:00] } 2... cxd4 { [%clk 0:01:59] } 3. Qxd4 { [%clk 0:02:01] } 3... Nc6 { [%clk 0:01:59] } 4. Qd1 { [%clk 0:02:01] } 4... Nf6 { [%clk 0:01:59] } 5. Nc3 { [%clk 0:02:01] } 5... d6 { [%clk 0:01:58] } 6. Nf3 { [%clk 0:01:59] } 6... B
[Event "Rated Bullet game"]
[Site "https://lichess.org/vca9O6Eb"]


## 2. Extraction

`read_games` (dans `src/utils/pgn.py`) lit les parties une par une, garde celles des cadences choisies, puis :
- chaque en-tête devient une colonne (Elo convertis en entiers) ;
- la ligne de coups devient `moves` = liste de `{san, clk, eval}` (`eval` = null si absente) ;
- `n_moves` = nombre de demi-coups.

In [4]:
t0 = time.time()
games = read_games(PGN_PATH, n_read=N_READ, time_controls=CADENCES)
games.write_parquet(RAW_PATH)
print(f"{games.height} parties gardées sur {N_READ} lues, en {time.time() - t0:.0f} s")
print(f"{RAW_PATH} : {RAW_PATH.stat().st_size / 1e6:.0f} Mo")

557237 parties gardées sur 2000000 lues, en 237 s
..\Data\blitz_180_raw.parquet : 168 Mo


## 3. Vérification rapide

In [5]:
games.schema

Schema([('Event', String),
        ('Site', String),
        ('Date', String),
        ('Round', String),
        ('White', String),
        ('Black', String),
        ('Result', String),
        ('UTCDate', String),
        ('UTCTime', String),
        ('WhiteElo', Int32),
        ('BlackElo', Int32),
        ('WhiteRatingDiff', Int32),
        ('BlackRatingDiff', Int32),
        ('ECO', String),
        ('Opening', String),
        ('TimeControl', String),
        ('Termination', String),
        ('WhiteTitle', String),
        ('BlackTitle', String),
        ('moves',
         List(Struct({'san': String, 'clk': String, 'eval': String}))),
        ('n_moves', UInt32)])

In [6]:
games.select("White", "WhiteElo", "Black", "BlackElo", "Result", "TimeControl", "Termination", "n_moves").head()

White,WhiteElo,Black,BlackElo,Result,TimeControl,Termination,n_moves
str,i32,str,i32,str,str,str,u32
"""DOMI60FR""",1735,"""Holizox""",1664,"""1/2-1/2""","""180+2""","""Normal""",99
"""Q8KSA""",1991,"""IRS007""",2045,"""1-0""","""180+0""","""Normal""",85
"""MiltonMorales""",2224,"""Glaidel""",2217,"""0-1""","""180+0""","""Normal""",28
"""nallek""",1744,"""Botonems""",1649,"""1-0""","""180+0""","""Time forfeit""",65
"""evgeniy_duk""",1557,"""Juanca_Master_Mind""",1611,"""0-1""","""180+2""","""Normal""",70


In [7]:
# Les 5 premiers coups de la 1re partie
games["moves"][0][:5].to_list()

[{'san': 'd4', 'clk': '0:03:00', 'eval': None},
 {'san': 'd5', 'clk': '0:03:00', 'eval': None},
 {'san': 'c4', 'clk': '0:03:01', 'eval': None},
 {'san': 'e5', 'clk': '0:02:59', 'eval': None},
 {'san': 'cxd5', 'clk': '0:03:01', 'eval': None}]

In [8]:
games["TimeControl"].value_counts(sort=True)

TimeControl,count
str,u32
"""180+0""",378584
"""180+2""",178653
